# Modèle UN : français + kirundi (Kaggle)

On **continue le modèle zéro** (on ne repart pas de rien) : même tokenizer, mêmes connaissances,
puis il relit l'ancien et lit le nouveau (`dataset_un` : textes kirundi vérifiés, insiguro).

| Étape | Ce qui se passe |
|---|---|
| 0 | Préparer : trouver les dossiers, l'heure de fin de session |
| 1 | Données publiques (Wikipédia FR...) : reprises du zéro, ou retéléchargées |
| 2 | **Relecture** : ancien (moins de Bible) + nouveau |
| 3 | **Discussion** : répondre, traduire, insiguro |
| 4 | Tests (les mêmes que le zéro + FLORES) |
| 5 | Essayer le modèle, puis le `.zip` |

**Sauvegardes** : petite sauvegarde tous les 50 pas, **copie complète toutes les 4 h**
(`sauvegarde_4h`, `sauvegarde_8h`), et **pause propre avant la fin de la session**.
Si une étape n'est pas finie à la pause, la **version suivante** du notebook reprend au même pas.

**Avant de lancer (Inputs)** : **Add Input** 3 fois :
1. Datasets > **modele-train** (l'ancien dataset du zéro)
2. Datasets > **dataset-un** (le nouveau)
3. Notebooks > **zero_kaggle** (sa sortie contient `modele_zero/lecture/finale`)

**Settings** : Accelerator = **GPU T4 x2**, Internet = **On**. Puis **Save Version > Save & Run All**.

**Session de 12 h** : le notebook s'arrête de lui-même vers 11 h (pause propre), les fichiers restent
dans l'onglet **Output**. **Pour continuer** : Add Input > Notebooks > **ce notebook (un_kaggle)**,
puis Save & Run All encore une fois : il recopie `modele_un` et reprend.

## Étape 0 : préparer Kaggle

In [ ]:
import glob, os, shutil, time
DEBUT = time.time()
HEURES_SESSION = 11.25     # Kaggle coupe à 12 h : on garde du temps pour les tests et la sortie
!nvidia-smi --query-gpu=name,memory.total --format=csv

def trouver(motif, quoi):
    t = sorted(glob.glob(motif, recursive=True))
    if not t:
        raise SystemExit(f"Introuvable : {quoi}. Vérifie les 'Inputs' (voir en haut).")
    return t[0]

DATASET = os.path.dirname(os.path.dirname(trouver("/kaggle/input/**/dialogues/salutations_rn.jsonl", "ancien dataset")))
DATASET_UN = os.path.dirname(trouver("/kaggle/input/**/insiguro/insiguro.txt", "dataset-un"))
ZERO = os.path.dirname(os.path.dirname(trouver("/kaggle/input/**/modele_zero/lecture/finale/config.json",
                                               "sortie du notebook zero_kaggle")))
TRAVAIL = "/kaggle/working/modele_un"
CODE = "/kaggle/working/code_un"
shutil.copytree(f"{DATASET_UN}/code", CODE, dirs_exist_ok=True)

# Reprise : sortie d'une version précédente de CE notebook ajoutée en Input
if not os.path.exists(TRAVAIL):
    anciens = sorted(glob.glob("/kaggle/input/**/modele_un/lecture", recursive=True))
    if anciens:
        print("Reprise depuis", os.path.dirname(anciens[0]), "(copie, quelques minutes)")
        shutil.copytree(os.path.dirname(anciens[0]), TRAVAIL)
PUBLICS = f"{ZERO}/publics" if glob.glob(f"{ZERO}/publics/lecture_fr/wikipedia_fr_001.txt") else f"{TRAVAIL}/publics"

In [ ]:
# 0.b  Commun : variables lues par train.py, heure de fin, état de chaque étape
import torch
os.makedirs(f"{TRAVAIL}/journaux", exist_ok=True)
J = f"{TRAVAIL}/journaux"
os.environ.update(DATASET=DATASET, DATASET_UN=DATASET_UN, ZERO=ZERO, TRAVAIL=TRAVAIL, PUBLICS=PUBLICS)
if HEURES_SESSION:
    os.environ["FIN_SESSION"] = str(DEBUT + HEURES_SESSION * 3600)
    print("Pause prévue vers", time.strftime("%H:%M", time.localtime(DEBUT + HEURES_SESSION * 3600 - 25 * 60)))
NB_GPU = max(1, torch.cuda.device_count())
LANCER = f"torchrun --nproc_per_node {NB_GPU} train.py" if NB_GPU > 1 else "python -u train.py"
print(f"{NB_GPU} GPU -> {LANCER}")
os.chdir(CODE)
PAUSE = False

def fait(chemin):
    return os.path.exists(chemin)

def apres(etape, dossier):
    """Après une étape : finie, en pause (reprise à la version suivante), ou en échec."""
    global PAUSE
    if fait(f"{dossier}/finale/config.json"):
        print(f"OK : {etape} terminée -> {dossier}/finale")
    elif fait(f"{dossier}/PAUSE"):
        PAUSE = True
        print(f"PAUSE : {etape} pas finie (temps de session). Les étapes suivantes attendent la version suivante.")
    else:
        raise SystemExit(f"ÉCHEC : {etape}. Regarde {J}/{etape}.txt")

def meilleur_modele():
    for ph in ("chat", "lecture"):
        for sous in ("finale", "meilleure"):
            if fait(f"{TRAVAIL}/{ph}/{sous}/config.json"):
                return f"{TRAVAIL}/{ph}/{sous}"
    return f"{ZERO}/lecture/finale"

print("DATASET    =", DATASET); print("DATASET_UN =", DATASET_UN); print("ZERO       =", ZERO)
print("PUBLICS    =", PUBLICS); print("TRAVAIL    =", TRAVAIL)
for nom, ph in [("Relecture", "lecture"), ("Discussion", "chat")]:
    d = f"{TRAVAIL}/{ph}"
    print(f"{nom:11s}: " + ("FINI" if fait(f"{d}/finale") else "en cours (reprise)" if fait(f"{d}/derniere")
                             else "pas commencé"))

## Étape 1 : données publiques
Reprises de la sortie du zéro si elles y sont, sinon téléchargées (1 fichier Wikipédia suffit).

In [ ]:
os.chdir(CODE)
if not glob.glob(f"{PUBLICS}/lecture_fr/wikipedia_fr_001.txt"):
    !python -u collecte/telecharger_publics.py --sortie {PUBLICS} --mots 20000000 2>&1 | grep -v "examples/s" | tee {J}/publics.txt
!ls {PUBLICS}/lecture_fr | head -3; ls {PUBLICS}/dialogues

## Étape 2 : relecture (continue le modèle zéro)
Regarde les phrases test : moins de répétitions et moins de style biblique qu'avec le zéro.
La dernière (« Guhodoka : ») montre s'il retient les insiguro.

In [ ]:
os.chdir(CODE)
if not PAUSE and not fait(f"{TRAVAIL}/lecture/finale"):
    !{LANCER} --config configs/un_lecture.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a {J}/lecture.txt
apres("lecture", f"{TRAVAIL}/lecture")

## Étape 3 : discussion (répondre, traduire, insiguro)

In [ ]:
os.chdir(CODE)
if not PAUSE and not fait(f"{TRAVAIL}/chat/finale"):
    !{LANCER} --config configs/un_chat.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a {J}/chat.txt
if not PAUSE:
    apres("chat", f"{TRAVAIL}/chat")

## Étape 4 : tests
Deux passages : **sans pénalité** (comparable au modèle zéro) et **avec pénalité de répétition 1.2**
(contre les boucles). Les catégories **« NON (vraie preuve) »** comptent : jamais vues à l'entraînement.
**flores_rn_fr / flores_fr_rn** : phrases traduites par des professionnels, hors Bible (si FLORES est dans dataset_un).

In [ ]:
os.chdir(CODE)
if PAUSE:
    print("Tests sautés : l'entraînement n'est pas fini (version suivante).")
else:
    tests = "tests/dialogues_test.jsonl" + (f" {DATASET_UN}/tests/flores.jsonl" if fait(f"{DATASET_UN}/tests/flores.jsonl") else "")
    !python -u evaluer.py --modele {meilleur_modele()} --tests {tests} --max 100 --montrer 3 \
        --sortie {J}/resultats_tests.jsonl 2>&1 | tee {J}/notes_tests.txt
    !python -u evaluer.py --modele {meilleur_modele()} --tests {tests} --max 100 --montrer 0 --penalite 1.2 \
        --sortie {J}/resultats_tests_penalite.jsonl 2>&1 | tee {J}/notes_tests_penalite.txt

## Étape 5 : essayer le modèle

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
appareil = "cuda" if torch.cuda.is_available() else "cpu"
dossier = meilleur_modele()
tok = AutoTokenizer.from_pretrained(dossier)
model = AutoModelForCausalLM.from_pretrained(dossier).to(appareil).eval()
historique = ""

def demander(question, garder=True):
    """Pose une question. garder=True : le modèle se souvient des tours d'avant."""
    global historique
    prompt = f"{historique}<|user|> {question} <|assistant|>"
    ids = tok(prompt, return_tensors="pt", return_token_type_ids=False).to(appareil)
    out = model.generate(**ids, max_new_tokens=150, do_sample=True, temperature=0.3, top_p=0.9,
                         repetition_penalty=1.2, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    reponse = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if garder:
        historique = f"{prompt} {reponse}<|endoftext|>"
    return reponse

print(dossier)
for q in ["Bonjour", "Bwakeye.", "N'amaki?", "Comment dit-on merci en kirundi ?", "Qui es-tu ?",
          "Traduis en français : Umwana n'umukobwa.", "Que veut dire « Guhodoka » en kirundi ?",
          "Que veut dire « Impingwe » en kirundi ?"]:
    print(q, "->", demander(q, garder=False))

## Étape 6 : le `.zip` pour le backend
À mettre dans `litgpt/models/local/`, puis `python scripts/convert_local_model.py <dossier>`.

In [ ]:
if PAUSE:
    print("PAS FINI. Pour continuer : Add Input > Notebooks > un_kaggle (cette version), puis Save & Run All.")
else:
    dossier_modele = meilleur_modele()
    !cd "{dossier_modele}" && zip -qr /kaggle/working/modele_un_fr_rn.zip . -x "etat_entrainement.pt"
    !ls -lh /kaggle/working/modele_un_fr_rn.zip
    print("Le .zip est dans l'onglet Output de cette version.")
print(f"Durée de cette session : {(time.time() - DEBUT) / 3600:.1f} h")